In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

# 1. 策略网络（简单 MLP）
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.ReLU(),
            nn.Linear(128, act_dim)
        )
    
    def forward(self, s):
        return torch.softmax(self.net(s), dim=-1)
    
    def act(self, s):
        probs = self.forward(s)
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        return a.item(), dist.log_prob(a)

# 2. REINFORCE 主循环
env = gym.make("CartPole-v1")
policy = PolicyNet(4, 2)
optimizer = optim.Adam(policy.parameters(), lr=1e-3)
gamma = 0.99

for episode in range(1000):
    log_probs = []
    rewards = []
    s, _ = env.reset()
    
    # 采样一条轨迹
    done = False
    while not done:
        a, log_prob = policy.act(torch.FloatTensor(s))
        s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        log_probs.append(log_prob)
        rewards.append(r)
    
    # 计算 discounted returns G_t
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)  # prepend
    
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-9)  # ← baseline: normalized return

    # 构造损失：-logπ * G_t （最小化 = 最大化 J(θ)）
    policy_loss = []
    for log_prob, G in zip(log_probs, returns):
        policy_loss.append(-log_prob * G)
    
    optimizer.zero_grad()
    loss = torch.stack(policy_loss).sum()
    loss.backward()
    optimizer.step()

    if episode % 50 == 0:
        print(f"Episode {episode}, Total Reward: {sum(rewards):.1f}")

Episode 0, Total Reward: 11.0
Episode 50, Total Reward: 22.0
Episode 100, Total Reward: 97.0
Episode 150, Total Reward: 63.0
Episode 200, Total Reward: 69.0
Episode 250, Total Reward: 174.0
Episode 300, Total Reward: 209.0
Episode 350, Total Reward: 369.0
Episode 400, Total Reward: 358.0
Episode 450, Total Reward: 309.0
Episode 500, Total Reward: 226.0
Episode 550, Total Reward: 500.0
Episode 600, Total Reward: 500.0
Episode 650, Total Reward: 290.0
Episode 700, Total Reward: 490.0
Episode 750, Total Reward: 500.0
Episode 800, Total Reward: 500.0
Episode 850, Total Reward: 500.0
Episode 900, Total Reward: 500.0
Episode 950, Total Reward: 500.0
